# A.4 答案能回到來源嗎？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**引用了真的文件，卻說文件裏沒有的日期，算可靠嗎？

給每片段一個來源ID，答案引用時能回到原文。引用存在只是第一步，真正要檢查來源內容支持答案。

**把直覺寫成數學：**citation validity與entailment分開；有效ID不保證陳述被支持。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
source = {"id": "d2", "text": "書店地址是青街8號"}
answer = {"text": "新地址青街8號", "citation": "d2"}
valid = answer["citation"] == source["id"]
supports = "青街8號" in source["text"]
print("來源存在", valid, "示例證據支持", supports)

**如何讀結果：**這個字符串檢查只適用窄示例；開放答案需人工或更完整驗證，不靠substring保證語意。

**只改一件事：**只把答案地址改爲紅街9號，保持相同引用ID。
